# 01 - Data Exploration

**Purpose**: Exploratory data analysis and data profiling for churn prediction project.

**Spec Reference**: `specs/001-churn-prediction-model/spec.md`

## Objectives
1. Profile customer and activity data
2. Understand data distributions and quality issues
3. Identify potential features for churn prediction
4. Document baseline statistics

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Project imports
import sys
sys.path.append('..')
from src.utils.config import get_config
from src.utils.logging import setup_logging, get_logger

# Setup
setup_logging()
logger = get_logger(__name__)
config = get_config()

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"History Window: {config.data.HISTORY_MONTHS} months")

## 1. Load Data from Lakehouse

Load customer master data and activity data from Fabric Lakehouse.

In [ ]:
# In Fabric, load from Lakehouse Delta tables
# Uncomment and modify paths for your Lakehouse

# customers_df = spark.read.format("delta").load("Tables/customers").toPandas()
# activity_df = spark.read.format("delta").load("Tables/customer_activity").toPandas()
# churn_events_df = spark.read.format("delta").load("Tables/churn_events").toPandas()

# For local development, create sample data
# TODO: Replace with actual data loading

print("Data loading - configure Lakehouse path above")

## 2. Customer Data Profile

In [ ]:
# Customer data overview
# Uncomment when data is loaded

# print(f"Total Customers: {len(customers_df):,}")
# print(f"Active Customers: {customers_df['is_active'].sum():,}")
# print(f"\nColumn Info:")
# print(customers_df.info())
# print(f"\nMissing Values:")
# print(customers_df.isnull().sum())

In [ ]:
# Customer segment distribution
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Segment distribution
# customers_df['segment'].value_counts().plot(kind='bar', ax=axes[0])
# axes[0].set_title('Customer Segments')
# axes[0].set_xlabel('Segment')
# axes[0].set_ylabel('Count')

# # Contract type distribution
# customers_df['contract_type'].value_counts().plot(kind='bar', ax=axes[1])
# axes[1].set_title('Contract Types')
# axes[1].set_xlabel('Contract Type')
# axes[1].set_ylabel('Count')

# plt.tight_layout()
# plt.show()

## 3. Activity Data Profile

In [ ]:
# Activity data overview
# print(f"Total Activity Records: {len(activity_df):,}")
# print(f"Unique Customers with Activity: {activity_df['customer_id'].nunique():,}")
# print(f"\nDate Range: {activity_df['activity_date'].min()} to {activity_df['activity_date'].max()}")
# print(f"\nActivity Types:")
# print(activity_df['activity_type'].value_counts())

In [ ]:
# Activity over time
# activity_df['activity_date'] = pd.to_datetime(activity_df['activity_date'])
# activity_df['activity_month'] = activity_df['activity_date'].dt.to_period('M')

# fig, ax = plt.subplots(figsize=(14, 5))
# activity_df.groupby('activity_month').size().plot(ax=ax)
# ax.set_title('Activity Volume Over Time')
# ax.set_xlabel('Month')
# ax.set_ylabel('Activity Count')
# plt.show()

## 4. Churn Analysis - Inactivity Buckets

Analyze potential churn based on inactivity thresholds:
- **>30 days**: Early warning
- **>60 days**: At risk
- **>90 days**: Likely churned

In [ ]:
# Calculate days since last activity per customer
# reference_date = datetime.now()

# last_activity = activity_df.groupby('customer_id')['activity_date'].max().reset_index()
# last_activity.columns = ['customer_id', 'last_activity_date']
# last_activity['days_inactive'] = (reference_date - last_activity['last_activity_date']).dt.days

# # Merge with customers
# customer_inactivity = customers_df.merge(last_activity, on='customer_id', how='left')

# # Calculate churn buckets
# bucket_30 = (customer_inactivity['days_inactive'] > 30).sum()
# bucket_60 = (customer_inactivity['days_inactive'] > 60).sum()
# bucket_90 = (customer_inactivity['days_inactive'] > 90).sum()

# print(f"Inactivity Analysis (as of {reference_date.date()})")
# print(f"="*50)
# print(f">30 days inactive: {bucket_30:,} ({bucket_30/len(customer_inactivity)*100:.1f}%)")
# print(f">60 days inactive: {bucket_60:,} ({bucket_60/len(customer_inactivity)*100:.1f}%)")
# print(f">90 days inactive: {bucket_90:,} ({bucket_90/len(customer_inactivity)*100:.1f}%)")

In [ ]:
# Inactivity distribution
# fig, ax = plt.subplots(figsize=(12, 5))
# customer_inactivity['days_inactive'].clip(upper=180).hist(bins=50, ax=ax)
# ax.axvline(x=30, color='yellow', linestyle='--', label='30 days')
# ax.axvline(x=60, color='orange', linestyle='--', label='60 days')
# ax.axvline(x=90, color='red', linestyle='--', label='90 days')
# ax.set_title('Distribution of Days Since Last Activity')
# ax.set_xlabel('Days Inactive')
# ax.set_ylabel('Customer Count')
# ax.legend()
# plt.show()

## 5. Explicit Churn Events

In [ ]:
# Explicit churn analysis
# print(f"Total Explicit Churn Events: {len(churn_events_df):,}")
# print(f"\nChurn by Reason Category:")
# print(churn_events_df['churn_reason_category'].value_counts())

## 6. Feature Candidates

Based on data exploration, identify candidate features for churn prediction:

In [ ]:
# Feature candidate analysis
feature_candidates = {
    'Customer Attributes': [
        'tenure_days - Days since start_date',
        'segment - Customer segment (one-hot encoded)',
        'region - Geographic region',
        'contract_type - Monthly/Annual/Multi-year',
        'contract_value - Annual contract value'
    ],
    'Activity Metrics': [
        'days_since_last_activity - Key predictor',
        'total_activities - Overall engagement',
        'activity_frequency - Activities per month',
        'login_count_30d/60d/90d - Recency patterns',
        'purchase_count - Commercial engagement',
        'support_ticket_count - May indicate issues'
    ],
    'Derived Features': [
        'activity_trend - Increasing/decreasing activity',
        'engagement_score - Composite metric',
        'time_to_first_activity - Onboarding success'
    ]
}

print("Feature Candidates for Churn Model")
print("="*50)
for category, features in feature_candidates.items():
    print(f"\n{category}:")
    for f in features:
        print(f"  - {f}")

## 7. Data Quality Summary

In [ ]:
# Data quality summary
# TODO: Fill after loading actual data

data_quality_summary = {
    'customers': {
        'row_count': 'TBD',
        'null_ratio': 'TBD',
        'duplicate_ids': 'TBD',
        'date_range': 'TBD'
    },
    'customer_activity': {
        'row_count': 'TBD',
        'null_ratio': 'TBD',
        'date_range': 'TBD',
        'coverage': 'TBD - % of customers with activity'
    },
    'churn_events': {
        'row_count': 'TBD',
        'explicit_count': 'TBD',
        'reason_coverage': 'TBD - % with reason provided'
    }
}

print("Data Quality Summary")
print("="*50)
for table, metrics in data_quality_summary.items():
    print(f"\n{table}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value}")

## 8. Baseline Statistics

Document baseline statistics for future comparison:

In [ ]:
# Baseline statistics to track
baseline_stats = {
    'analysis_date': datetime.now().isoformat(),
    'total_customers': 'TBD',
    'active_customers': 'TBD',
    'churn_rate_30d': 'TBD',
    'churn_rate_60d': 'TBD',
    'churn_rate_90d': 'TBD',
    'churn_rate_explicit': 'TBD',
    'avg_tenure_days': 'TBD',
    'avg_monthly_activity': 'TBD',
    'avg_contract_value': 'TBD'
}

print("Baseline Statistics")
print("="*50)
for stat, value in baseline_stats.items():
    print(f"{stat}: {value}")

# Save baseline to file
# import json
# with open('../data/baseline_stats.json', 'w') as f:
#     json.dump(baseline_stats, f, indent=2)

## Next Steps

1. **Configure Lakehouse** - Update data loading cells with actual Lakehouse paths
2. **Run full profiling** - Execute all cells with real data
3. **Document issues** - Record any data quality issues found
4. **Proceed to feature engineering** - `02_feature_engineering.ipynb`